# Analyse exploratoire du marché Steam - Projet Ubisoft

### Contexte et objectifs
Ubisoft souhaite faire une analyse approfondie du marché du jeux vidéo à partir de Steam. Ce notebook réalise une analyse exploratoire (EDA) du catalogue Steam (~55 700 jeux) pour identifier les tendances et facteurs clés de succès.

In [0]:
# Test avant de commencer
print(f"La version de Spark: {spark.version}")

La version de Spark: 4.2.0


In [0]:
# Chargement des données Steam récupérées depuis S3 
file_path = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
df = spark.read.json(file_path)

# Afficher le schéma
df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:
df.show(5)

+--------------------+-------+
|                data|     id|
+--------------------+-------+
|{10, [Multi-playe...|     10|
|{1000000, [Single...|1000000|
|{1000010, [Single...|1000010|
|{1000030, [Multi-...|1000030|
|{1000040, [Single...|1000040|
+--------------------+-------+
only showing top 5 rows


In [0]:
print(f"Dimensions du dataset: {(df.count(), len(df.columns))}")

Dimensions du dataset: (55691, 2)


In [0]:
df.columns

['data', 'id']

In [0]:
from pyspark.sql import functions as F

# On désimbrique/aplatit le datset et ainsi extraire les colonnes nécessaires à l'analyse
df_flat = df.select(
    F.col("id"),
    F.col("data.appid").alias("appid"),
    F.col("data.name").alias("name"),
    F.col("data.developer").alias("developer"),
    F.col("data.publisher").alias("publisher"),
    F.col("data.genre").alias("genre"),
    F.col("data.release_date").alias("release_date"),
    F.col("data.owners").alias("owners"),
    F.col("data.positive").alias("positive"),
    F.col("data.negative").alias("negative"),
    F.col("data.price").alias("price"),
    F.col("data.discount").alias("discount"),
    F.col("data.required_age").alias("required_age"),
    F.col("data.languages").alias("languages"),
    F.col("data.categories").alias("categories"),
    F.col("data.platforms.windows").alias("windows"),
    F.col("data.platforms.mac").alias("mac"),
    F.col("data.platforms.linux").alias("linux"),
    F.col("data.ccu").alias("ccu"),
    F.col("data.type").alias("type"),
)

# Vérification
print("Nombre de jeux :", df_flat.count())
df_flat.show(5)

Nombre de jeux : 55691
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+--------+--------+-----+--------+------------+--------------------+--------------------+-------+-----+-----+-----+----+
|     id|  appid|                name|           developer|           publisher|               genre|release_date|              owners|positive|negative|price|discount|required_age|           languages|          categories|windows|  mac|linux|  ccu|type|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+--------+--------+-----+--------+------------+--------------------+--------------------+-------+-----+-----+-----+----+
|     10|     10|      Counter-Strike|               Valve|               Valve|              Action|   2000/11/1|10,000,000 .. 20,...|  201215|    5199|  999|       0|           0|English, French, ...|[Multi-pla

In [0]:
# Nettoyage
# Garder uniquement les vrais jeux de type "game"
df_games = df_flat.filter(F.col("type") == "game")

# Formatter la colonne "release_date" au type date
df_games = df_games.withColumn(
    "release_date",
    F.when(
        F.col("release_date").rlike(r"^\d{4}/\d{1,2}/\d{1,2}$"),
        F.to_date(F.col("release_date"), "yyyy/M/d")
    ).when(
        F.col("release_date").rlike(r"^\d{4}/\d{1,2}$"),
        F.to_date(F.col("release_date"), "yyyy/M")
    ).when(
        F.col("release_date").rlike(r"^\d{4}$"),
        F.to_date(F.col("release_date"), "yyyy")
    )
)

# Convertir le prix en $ car il est en centimes
df_games = df_games.withColumn("price", F.col("price")/100)

# Convertir required_age en nombre entier
df_games = df_games.withColumn("required_age", F.regexp_extract("required_age", r"(\d+)", 1).cast("int"))

# Créer une colonne ratio d'avis positifs (%)
df_games = df_games.withColumn(
    "positive_ratio",
    F.when(
        (F.col("positive") + F.col("negative")) > 0,
        F.round(F.col("positive") / (F.col("positive") + F.col("negative")) * 100, 2)
    ).otherwise(None)
)

# Vérification
print("Nombre de jeux après nettoyage :", df_games.count())
df_games.select("name", "price", "required_age", "positive_ratio").show(5)
                                

Nombre de jeux après nettoyage : 55690
+--------------------+-----+------------+--------------+
|                name|price|required_age|positive_ratio|
+--------------------+-----+------------+--------------+
|      Counter-Strike| 9.99|           0|         97.48|
|           ASCENXION| 9.99|           0|         84.38|
|         Crown Trick| 5.99|           0|         86.19|
|Cook, Serve, Deli...|19.99|           0|          93.2|
|            细胞战争| 1.99|           0|           0.0|
+--------------------+-----+------------+--------------+
only showing top 5 rows


In [0]:
# ===================================================================================================
# Question 1 : Quel éditeur a sorti le plus de jeux sur Steam ??
# ===================================================================================================

df_publishers = df_games.filter(F.col("publisher") != "") \
    .groupBy("publisher") \
    .agg(F.count("appid").alias("nombre_jeux")) \
        .orderBy(F.desc("nombre_jeux"))

# Visualisation : Top des 10 éditeurs
display(df_publishers.limit(10))

publisher,nombre_jeux
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
Sekai Project,132
HH-Games,132
Ubisoft,127
Laush Studio,126


Databricks visualization. Run in Databricks to view.

In [0]:
# =================================================================================================
# Question 2 : Quels sont les jeux les mieux notés ?
# minimum 100 avis pour que cela soit représentatif
# =================================================================================================

df_best_games = df_games.filter(F.col("positive") + F.col("negative") > 100) \
    .orderBy(F.desc("positive_ratio"), F.desc("positive")) \
    .select("name", "publisher", "positive", "negative", "positive_ratio")
            
display(df_best_games.limit(10))


name,publisher,positive,negative,positive_ratio
The Void Rains Upon Her Heart,The Hidden Levels,496,0,100.0
祈風 Inorikaze,觀象草圖 Astrolabe Draft,327,0,100.0
秘封旅行 ~ Secret Sealing Travel,鸽屋谷,218,0,100.0
Elasto Mania Remastered,Elasto Mania Team,190,0,100.0
Freshly Frosted,The Quantum Astrophysicists Guild,157,0,100.0
HAYAI,Chaoclypse,148,0,100.0
FIND ALL 2: Middle Ages,Very Very LITTLE Studio,132,0,100.0
Touhou Kaeizuka ～ Phantasmagoria of Flower View.,"Mediascape Co., Ltd.",119,0,100.0
Lucy Dreaming,Tall Story Games Ltd,118,0,100.0
未来战士,BIFROST CLOUD PTE. LTD.,116,0,100.0


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import year
# =================================================================================================
# Question 3 : y a-t-il eu des années avec plus de sorties de jeux ?
# =================================================================================================

df_years = df_games.withColumn("year", year("release_date")) \
.groupBy("year") \
.agg(F.count("appid").alias("nombre_sorties")) \
.orderBy("year")

display(df_years)

year,nombre_sorties
null,99
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6


Databricks visualization. Run in Databricks to view.

In [0]:
# =================================================================================================
# Question 4 : Comment les prix sont-ils distribués ?
# =================================================================================================

# Distribution des prix :
df_prix = df_games.select(
    F.when(F.col("price") < 5, "moins de 5$")
     .when(F.col("price") < 10, "5$ à 10$")
     .when(F.col("price") < 20, "10$ à 20$")
     .when(F.col("price") < 50, "20$ à 50$")
     .otherwise("Plus de 50$")
     .alias("tranche")
).groupBy("tranche").count().orderBy(F.col("count").desc())

df_prix_describe = df_games.select(F.col("price")).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")

# Affichage
display(df_prix)
display(df_prix_describe)

tranche,count
moins de 5$,31257
5$ à 10$,12450
10$ à 20$,9022
20$ à 50$,2633
Plus de 50$,328


Databricks visualization. Run in Databricks to view.

summary,price
count,55690
mean,7.732988687371752
stddev,10.931394858517274
min,0.0
25%,1.34
50%,4.99
75%,9.99
max,999.0


In [0]:
# =================================================================================================
# Question 5 : Quelles sont les langues les plus représentées ?
# =================================================================================================

df_languages = df_games.filter(F.col("languages").isNotNull()) \
    .withColumn("language", F.explode(F.split(F.col("languages"), ", "))) \
    .groupBy("language") \
    .agg(F.count("appid").alias("nombre_jeux")) \
    .orderBy(F.desc("nombre_jeux"))

display(df_languages.limit(15))

language,nombre_jeux
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6599


Databricks visualization. Run in Databricks to view.

In [0]:
# ==============================================================================================
# Question 6 : Existe-t-il beaucoup de jeux interdits aux moins de 16/18 ans ?
# ==============================================================================================

df_age = df_games.select(
    F.when(F.col("required_age") < 3, "moins de 3 ans")
     .when(F.col("required_age") < 7, "moins de 7 ans")
     .when(F.col("required_age") < 12, "moins de 12 ans")
     .when(F.col("required_age") < 16, "moins de 16 ans")
     .otherwise("moins de 18 ans")
     .alias("tranche")
).groupBy("tranche").count().orderBy(F.col("count").desc())

display(df_age)

tranche,count
moins de 3 ans,55029
moins de 16 ans,333
moins de 18 ans,306
moins de 12 ans,14
moins de 7 ans,8


Databricks visualization. Run in Databricks to view.

In [0]:
# =================================================================================================
# Question 7 : Quels sont les genres les plus représentés ?
# =================================================================================================

df_genres = (
    df_games.filter(F.col("genre").isNotNull())
    .withColumn("genre_split", F.explode(F.split(F.col("genre"), ", ")))
    .filter(F.col("genre_split") != "")
    .groupBy("genre_split")
    .agg(F.countDistinct("appid").alias("nombre_jeux"))
    .withColumn("pourcentage", F.round(F.col("nombre_jeux") * 100 / df_games.count(), 2))
    .orderBy(F.desc("nombre_jeux"))
)

display(df_genres)

genre_split,nombre_jeux,pourcentage
Indie,39681,71.25
Action,23759,42.66
Casual,22086,39.66
Adventure,21431,38.48
Strategy,10895,19.56
Simulation,10836,19.46
RPG,9534,17.12
Early Access,6145,11.03
Free to Play,3393,6.09
Sports,2666,4.79


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# =================================================================================================
# Question 8 : Quels genres ont le meilleur ratio d'avis positifs ?
# =================================================================================================

df_genres_ratio = (
    df_games.filter(F.col("genre").isNotNull())
    .withColumn("genre_split", F.explode(F.split(F.col("genre"), ", ")))
    .filter(F.col("genre_split") != "")
    .groupBy("genre_split")
    .agg(
        F.round(F.avg("positive_ratio"), 2).alias("ratio_moyen"),
        F.countDistinct("appid").alias("nombre_jeux")
    )
    .orderBy(F.desc("ratio_moyen"))
)

display(df_genres_ratio.select("genre_split", "ratio_moyen", "nombre_jeux"))

genre_split,ratio_moyen,nombre_jeux
Casual,74.34,22086
Indie,74.21,39681
Adventure,73.88,21431
Game Development,73.78,159
RPG,73.06,9534
Action,73.0,23759
Accounting,72.33,16
Free to Play,72.19,3393
Design & Illustration,72.07,406
Web Publishing,71.97,89


Databricks visualization. Run in Databricks to view.

In [0]:
#==================================================================================================
# Question 9 : Quels sont les genres les plus lucratifs ?
# =================================================================================================

# Ceci est une estimation à partir des personnes (le point médian des owners) qui ont obtenu le jeux et son prix
df_genres_income = (
    df_games
    .withColumn(
        "owners_min",
        F.regexp_extract("owners", r"([\d,]+)", 1)
    )
    .withColumn(
        "owners_min",
        F.regexp_replace("owners_min", ",", "").cast("long")
    )
    .withColumn(
        "owners_max",
        F.regexp_extract("owners", r"\.\.\s*([\d,]+)", 1)
    )
    .withColumn(
        "owners_max",
        F.regexp_replace("owners_max", ",", "").cast("long")
    )
    .withColumn(
        "owners_estim",
        (F.col("owners_min") + F.col("owners_max")) / 2
    )
    .withColumn(
        "revenu_estim",
        F.col("owners_estim") * F.col("price")
    )
)

df_genres_income = (
    df_genres_income.filter(F.col("genre").isNotNull() & F.col("owners").isNotNull())
    .withColumn("genre_split", F.explode(F.split(F.col("genre"), ", ")))
    .filter(F.col("genre_split") != "")
    .groupBy("genre_split")
    .agg(F.round(F.sum("revenu_estim"), 2).alias("total_revenu_estim"), F.countDistinct("appid").alias("nombre_jeux"))
    .orderBy(F.desc("total_revenu_estim"))
)

display(df_genres_income.limit(15))


genre_split,total_revenu_estim,nombre_jeux
Action,5.87564541E10,23759
Adventure,3.724573845E10,21431
Indie,3.23465771E10,39681
RPG,2.71731431E10,9534
Strategy,2.015004105E10,10895
Simulation,1.876974955E10,10836
Casual,8.08095665E9,22086
Massively Multiplayer,5.93015775E9,1460
Early Access,5.45866145E9,6145
Sports,3.14989735E9,2666


Databricks visualization. Run in Databricks to view.

In [0]:
# =================================================================================================
# Question 10 : La plupart des jeux sont-ils disponibles sur Windows/Mac/Linux ?
# =================================================================================================

df_platforms = spark.createDataFrame([
    ("Windows", df_games.filter(F.col("windows") == True).count()),
    ("Mac", df_games.filter(F.col("mac") == True).count()),
    ("Linux", df_games.filter(F.col("linux") == True).count())
], ["plateforme", "nombre_jeux"])

df_platforms = df_platforms.withColumn("pourcentage", F.round(F.col("nombre_jeux") * 100 / df_games.count(), 2))

display(df_platforms)

plateforme,nombre_jeux,pourcentage
Windows,55675,99.97
Mac,12769,22.93
Linux,8457,15.19


Databricks visualization. Run in Databricks to view.

In [0]:
# =================================================================================================
# Question 11 : Certains genres sont-ils disponibles de préférence sur certaines plateformes ?
# =================================================================================================

df_genre_platform = df_games.filter(F.col("genre").isNotNull()) \
    .withColumn("genre_split", F.explode(F.split(F.col("genre"), ", "))) \
    .filter(F.col("genre_split") != "")\
    .groupBy("genre_split") \
    .agg(
        F.round(F.avg(F.col("windows").cast("int")) * 100, 1).alias("pct_windows"),
        F.round(F.avg(F.col("mac").cast("int")) * 100, 1).alias("pct_mac"),
        F.round(F.avg(F.col("linux").cast("int")) * 100, 1).alias("pct_linux"),
        F.count("appid").alias("nombre_jeux")
    ) \
    .filter(F.col("nombre_jeux") >= 100) \
    .orderBy(F.desc("nombre_jeux"))

display(df_genre_platform)

genre_split,pct_windows,pct_mac,pct_linux,nombre_jeux
Indie,100.0,25.0,17.6,39681
Action,100.0,19.2,14.2,23759
Casual,100.0,23.2,15.0,22086
Adventure,100.0,23.5,15.4,21431
Strategy,100.0,27.6,16.8,10895
Simulation,100.0,22.5,14.1,10836
RPG,100.0,23.6,16.0,9534
Early Access,100.0,14.6,10.3,6145
Free to Play,99.9,24.9,14.0,3393
Sports,100.0,19.0,10.8,2666


Databricks visualization. Run in Databricks to view.